# 🎙️ ElevenLabs SOTA Russian Studio (Flow-Matching DiT + Vocos)
### Ультра-премиум клонирование живого голоса на **русском языке** без роботов и скрежета
---
> * 🔥 **Flow-Matching DiT**: Новейшая диффузионная архитектура (полное отсутствие металлического звука).
> * 🇷🇺 **5,000+ часов русской речи**: Специальная модель `Misha24-10/F5-TTS_RUSSIAN` с идеальной фонетикой.
> * 🔤 **RUAccent**: Автоматическая расстановка русских ударений на лету.
> * 📚 **VoiceLab Library**: Сохраняйте голоса 1 раз — они остаются в выпадающем списке навсегда.
> * 💾 **Google Drive**: Постоянное хранение всех созданных голосов в `MyDrive/ElevenLabs_Voices`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. ⚙️ Установка SOTA окружения
Нажмите **Play**. Занимает около 50 секунд.

In [ ]:
# @title ⚙️ 1. Установка SOTA Russian Diffusion Studio (F5-TTS + Vocos)
# Нажмите Play слева (установка займет ~40-50 секунд)

import os
import sys

print("🚀 [1/3] Подключение Google Drive для вечного сохранения голосов...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices'
    print(f"💾 Google Drive подключен! Ваши голоса сохраняются в: {VOICES_DIR}")
except Exception:
    VOICES_DIR = '/content/voices'
    print(f"💾 Работаем локально (голоса будут в {VOICES_DIR}).")

os.makedirs(VOICES_DIR, exist_ok=True)

print("📦 [2/3] Установка нейросетевых библиотек (F5-TTS, Vocos 24kHz, Whisper, RUAccent)...")
os.system("pip install -q f5-tts ruaccent openai-whisper soundfile torchaudio gradio pydub")
os.system("apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1")

print("\n============================================================")
print("✨ ВСЁ ГОТОВО! Переходите к Ячейке №2 ниже для запуска Студии!")
print("============================================================")


## 2. 🚀 Запуск премиум-студии
Нажмите **Play**. Через ~1.5 минуты появится ссылка **`Running on public URL: https://xxxx.gradio.live`**.
Откройте её на телефоне или компьютере.

In [ ]:
import numpy as np
import time
# @title 🚀 2. Запуск ElevenLabs Russian Studio (F5-TTS + VoiceLab)
# Нажмите Play слева для открытия интерфейса

import os
import re
import json
import shutil
import tempfile
import torch
import torchaudio
import soundfile as sf
import whisper
import gradio as gr
from pydub import AudioSegment
from huggingface_hub import hf_hub_download
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f"⚡ Устройство: {device} ({dev_name})")

VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices' if os.path.exists('/content/drive/MyDrive') else '/content/voices'
os.makedirs(VOICES_DIR, exist_ok=True)

print("📥 [1/4] Загрузка нейровокодера Vocos 24kHz (чистый звук без скрежета)...")
vocoder = load_vocoder(vocoder_name='vocos', device=device)

print("📥 [2/4] Загрузка русской модели F5-TTS Russian (5000+ часов русской речи)...")
ckpt_path = hf_hub_download(repo_id='Misha24-10/F5-TTS_RUSSIAN', filename='F5TTS_v1_Base_v2/model_last_inference.safetensors')
vocab_path = hf_hub_download(repo_id='Misha24-10/F5-TTS_RUSSIAN', filename='F5TTS_v1_Base/vocab.txt')
model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
model = load_model(DiT, model_cfg, ckpt_path=ckpt_path, vocab_file=vocab_path, device=device)

print("📥 [3/4] Загрузка Whisper ASR (автоматическая расшифровка сэмплов)...")
asr = whisper.load_model('base', device=device)

print("📥 [4/4] Инициализация RUAccent turbo3.1 (автоматическая расстановка русских ударений +)...")
from ruaccent import RUAccent
accentizer = RUAccent()
accentizer.load(omograph_model_size='turbo3.1', use_dictionary=True, tiny_mode=False)
print("✅ RUAccent turbo3.1 готов! Все слова размечаются знаками '+' перед ударными гласными.")

print("🔥 Премиум-студия F5-TTS готова к генерации живой русской речи!")

def get_saved_voices():
    voices = []
    if os.path.exists(VOICES_DIR):
        for d in sorted(os.listdir(VOICES_DIR)):
            v_dir = os.path.join(VOICES_DIR, d)
            if os.path.isdir(v_dir):
                has_wav = os.path.exists(os.path.join(v_dir, 'sample.wav')) or os.path.exists(os.path.join(v_dir, 'ref.wav'))
                if has_wav:
                    voices.append(d)
    return voices if voices else ['[Нет сохраненных голосов]']

def get_voice_files(voice_name):
    v_dir = os.path.join(VOICES_DIR, voice_name)
    wav_path = os.path.join(v_dir, 'sample.wav')
    if not os.path.exists(wav_path):
        wav_path = os.path.join(v_dir, 'ref.wav')
    
    txt_path = os.path.join(v_dir, 'ref_text.txt')
    if not os.path.exists(txt_path):
        txt_path = os.path.join(v_dir, 'ref.txt')
        
    return wav_path, txt_path

def on_voice_select(selected_voice):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        return None
    wav_path, _ = get_voice_files(selected_voice)
    return wav_path if os.path.exists(wav_path) else None

def load_any_audio(f):
    path = f.name if hasattr(f, 'name') else str(f)
    # 1. Torchaudio
    try:
        wav, sr = torchaudio.load(path)
        if sr != 24000:
            wav = torchaudio.functional.resample(wav, sr, 24000)
        if wav.shape[0] > 1:
            wav = torch.mean(wav, dim=0, keepdim=True)
        return wav
    except Exception:
        pass
    # 2. Soundfile
    try:
        data, sr = sf.read(path)
        if data.ndim > 1:
            data = data.mean(axis=1)
        wav = torch.tensor(data, dtype=torch.float32).unsqueeze(0)
        if sr != 24000:
            wav = torchaudio.functional.resample(wav, sr, 24000)
        return wav
    except Exception:
        pass
    # 3. Pydub / AudioSegment
    try:
        seg = AudioSegment.from_file(path)
        seg = seg.set_channels(1).set_frame_rate(24000)
        samples = np.array(seg.get_array_of_samples(), dtype=np.float32) / 32768.0
        return torch.tensor(samples, dtype=torch.float32).unsqueeze(0)
    except Exception:
        pass
    # 4. Direct FFmpeg conversion (любые OGG, OPUS, MP3, M4A)
    try:
        tmp_wav = tempfile.mktemp(suffix=".wav")
        os.system(f'ffmpeg -y -i "{path}" -ar 24000 -ac 1 "{tmp_wav}" > /dev/null 2>&1')
        if os.path.exists(tmp_wav) and os.path.getsize(tmp_wav) > 1000:
            wav, _ = torchaudio.load(tmp_wav)
            try:
                os.remove(tmp_wav)
            except Exception:
                pass
            return wav
    except Exception:
        pass
    return None

def save_new_voice(voice_name, audio_files, custom_text):
    try:
        if not voice_name or not voice_name.strip():
            return gr.update(), "⚠️ Введите название голоса (например: Мой голос, Есенин)!", None, ""
        if not audio_files:
            return gr.update(), "⚠️ Загрузите хотя бы один аудиофайл!", None, ""

        clean_name = re.sub(r'[^a-zA-Z0-9а-яА-ЯёЁ_\- ]', '', voice_name).strip()
        if not clean_name:
            clean_name = "Voice_" + str(int(time.time()))

        voice_folder = os.path.join(VOICES_DIR, clean_name)
        os.makedirs(voice_folder, exist_ok=True)
        sample_wav = os.path.join(voice_folder, 'sample.wav')

        files = audio_files if isinstance(audio_files, list) else [audio_files]
        wav_tensors = []
        for f in files:
            tensor = load_any_audio(f)
            if tensor is not None and tensor.shape[1] > 0:
                wav_tensors.append(tensor)

        if not wav_tensors:
            return gr.update(), "❌ Не удалось прочитать аудиозаписи. Проверьте формат файлов.", None, ""

        combined = torch.cat(wav_tensors, dim=1)
        # Ограничение длины эталона: 7-12 секунд (оптимально для F5-TTS)
        max_samples = 24000 * 12
        if combined.shape[1] > max_samples:
            start_s = max(0, (combined.shape[1] - (24000 * 10)) // 2)
            combined = combined[:, start_s:start_s + (24000 * 10)]

        # Нормализация громкости
        max_val = torch.max(torch.abs(combined))
        if max_val > 0:
            combined = combined / max_val * 0.95

        torchaudio.save(sample_wav, combined, 24000)

        # Распознавание текста через Whisper
        if custom_text and custom_text.strip():
            text_raw = custom_text.strip()
        else:
            print("📝 Whisper расшифровывает образец речи...")
            res = asr.transcribe(sample_wav, language='ru')
            text_raw = res.get('text', '').strip()

        if not text_raw:
            text_raw = "Привет, это образец моего голоса для клонирования речи."

        # Расстановка ударений
        try:
            text_accented = accentizer.process_all(text_raw)
        except Exception:
            text_accented = text_raw

        with open(os.path.join(voice_folder, 'ref_text.txt'), 'w', encoding='utf-8') as f:
            f.write(text_accented)

        new_list = get_saved_voices()
        status_msg = f"🎉 Голос '{clean_name}' успешно обучен и сохранен в Google Drive!"
        print(f"✅ Голос '{clean_name}' готов! Текст: {text_accented}")
        return gr.update(choices=new_list, value=clean_name), status_msg, sample_wav, text_raw

    except Exception as e:
        import traceback
        traceback.print_exc()
        return gr.update(), f"❌ Ошибка при добавлении голоса: {e}", None, ""


def generate_speech(selected_voice, text, speed):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        raise gr.Error('⚠️ Выберите голос из списка или добавьте новый во вкладке VoiceLab!')
    if not text or not text.strip():
        raise gr.Error('⚠️ Введите русский текст для озвучки!')

    wav_path, txt_path = get_voice_files(selected_voice)
    if not os.path.exists(wav_path):
        raise gr.Error(f'⚠️ Файл аудио для голоса "{selected_voice}" не найден!')

    ref_text_accented = ""
    if os.path.exists(txt_path):
        with open(txt_path, 'r', encoding='utf-8') as f:
            ref_text_accented = f.read().strip()

    if not ref_text_accented:
        res = asr.transcribe(wav_path, language='ru')
        ref_text_accented = accentizer.process_all(res.get('text', '').strip())

    # Автоматическая расстановка ударений в целевом русском тексте
    target_accented = accentizer.process_all(text.strip())
    print(f"🎙️ [F5-TTS SOTA] Полный текст с ударениями:\n{target_accented}\n")

    final_wave, final_sr, _ = infer_process(
        ref_audio=wav_path,
        ref_text=ref_text_accented,
        gen_text=target_accented,
        model_obj=model,
        vocoder=vocoder,
        speed=float(speed),
        device=device
    )

    out_file = tempfile.mktemp(suffix='.wav')
    sf.write(out_file, final_wave, final_sr)
    return out_file

initial_voices = get_saved_voices()
default_poem = '''Не жалею, не зову, не плачу,
Всё пройдет, как с белых яблонь дым.
Увяданья золотом охваченный,
Я не буду больше молодым.

Ты теперь не так уж будешь биться,
Сердце, тронутое холодком,
И страна березового ситца
Не заманит шляться босиком.'''

theme = gr.themes.Soft(primary_hue="amber", neutral_hue="slate")

with gr.Blocks(theme=theme, title="🎙️ ElevenLabs SOTA Voice Studio (F5-TTS Russian)") as demo:
    gr.Markdown("# 🎙️ ElevenLabs Professional Voice Studio (F5-TTS Russian SOTA)")
    gr.Markdown("*Премиальное нейросетевое клонирование голоса и генерация русской речи на базе Flow-Matching DiT (5000+ часов русской речи) + Vocos 24kHz.*")

    with gr.Tabs():
        with gr.Tab("🎙️ Студия озвучки (Studio)"):
            with gr.Row():
                with gr.Column(scale=1):
                    voice_dropdown = gr.Dropdown(
                        choices=initial_voices,
                        value=initial_voices[0] if initial_voices else None,
                        label="Выберите сохраненный голос",
                        interactive=True
                    )
                    voice_preview = gr.Audio(
                        value=on_voice_select(initial_voices[0]) if initial_voices else None,
                        label="🎵 Как звучит эталон голоса",
                        type="filepath",
                        interactive=False
                    )
                    speed_slider = gr.Slider(minimum=0.7, maximum=1.4, value=1.0, step=0.05, label="Скорость речи (Speed)")
                    refresh_btn = gr.Button("🔄 Обновить список голосов")

                with gr.Column(scale=2):
                    text_input = gr.Textbox(
                        lines=10,
                        value=default_poem,
                        label="Введите любой русский текст или стихи",
                        placeholder="Вставьте русский текст для озвучки..."
                    )
                    synth_btn = gr.Button("⚡ Озвучить выбранным голосом (F5-TTS SOTA)", variant="primary", size="lg")
                    audio_output = gr.Audio(label="🎧 Результат (Студийный WAV 24kHz)", type="filepath", interactive=False)

        with gr.Tab("🔬 VoiceLab (Обучение нового голоса)"):
            gr.Markdown("### 📂 Загрузите аудиозаписи для обучения нового голоса")
            gr.Markdown("> Загрузите 1 или несколько аудиофайлов (голосовые сообщения, видео, диктофон). Whisper расшифрует слова, RUAccent расставит ударения, а модель сохранит голос в Google Drive навсегда.")
            with gr.Row():
                with gr.Column():
                    new_voice_name = gr.Textbox(label="Название голоса (например, 'Мой Голос' или 'Есенин')")
                    audio_upload = gr.File(
                        file_count="multiple",
                        file_types=["audio"],
                        label="Загрузите аудиозаписи (WAV, MP3, M4A, OGG)"
                    )
                    custom_text_input = gr.Textbox(
                        label="Текст из аудио (необязательно, Whisper расшифрует сам)",
                        placeholder="Оставьте пустым для авто-распознавания..."
                    )
                    train_btn = gr.Button("🔥 Обучить и сохранить голос в библиотеку", variant="primary")
                with gr.Column():
                    train_status = gr.Markdown("")
                    train_preview = gr.Audio(label="Очищенный эталонный сэмпл голоса", type="filepath", interactive=False)
                    rec_text_display = gr.Textbox(label="Распознанный текст сэмпла", interactive=False)

    voice_dropdown.change(on_voice_select, inputs=[voice_dropdown], outputs=[voice_preview])
    refresh_btn.click(lambda: gr.update(choices=get_saved_voices()), outputs=[voice_dropdown])

    synth_btn.click(
        generate_speech,
        inputs=[voice_dropdown, text_input, speed_slider],
        outputs=[audio_output]
    )

    train_btn.click(
        save_new_voice,
        inputs=[new_voice_name, audio_upload, custom_text_input],
        outputs=[voice_dropdown, train_status, train_preview, rec_text_display]
    )

print("🚀 Запуск публичной ссылки...")
demo.queue().launch(share=True, debug=False)
